# THE BLACK BOX — Manim render notebook
### Calculus Chapter 1 · Functions and Models · 41 levels

Yeh notebook `scene.py` files ko **as-is** render karta hai — code duplicate nahi hota,
source of truth aapke folder me rehti hai.

**Zaroori baat:** scenes me koi LaTeX (`MathTex`) nahi hai — sab Pango `Text` hai.
Is liye TeXLive ka 2 GB install **nahi** chahiye. Setup ~90 seconds.

---

### Ek dafa ka kaam

1. Apne PC se `ch01-black-box` folder **Google Drive** me `MyDrive` ke andar upload karein
   (poora folder — `shared/` aur `L01_.../` sab).
2. Neeche Cell 1 (setup) aur Cell 2 (drive mount) chalayein.
3. Phir jo level render karna ho, uska cell chalayein.

Session reset hone par sirf Cell 1 aur 2 dobara chalane hain.

## Cell 1 — Setup (~90 sec, har naye session me ek baar)

In [ ]:
%%capture setup_log
!sudo apt-get update -qq
!sudo apt-get install -y -qq libcairo2-dev libpango1.0-dev ffmpeg
!pip install -q manim

In [ ]:
import manim, shutil
print('manim  :', manim.__version__)
print('ffmpeg :', shutil.which('ffmpeg'))
print('\nSetup theek hai — Cell 2 par jayein.')

## Cell 2 — Drive mount + project path

Agar Drive use nahi karna, to neeche wale cell ki jagah folder ka zip upload karein
(uska code sabse aakhir me 'Plan B' me hai).

In [ ]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/ch01-black-box')   # <- apna path adjust karein
assert PROJECT.exists(), f'Nahi mila: {PROJECT} — Drive me folder ka naam/jagah check karein'

sys.path.insert(0, str(PROJECT / 'shared'))
print('PROJECT :', PROJECT)
print('LEVELS  :', sorted(p.name for p in PROJECT.glob('L*_*') if p.is_dir()))

## Cell 3 — Render helper (ek baar chalayein)

In [ ]:
import subprocess, pathlib
from IPython.display import Video, display, HTML

QDIR = {'l': '480p15', 'm': '720p30', 'h': '1080p60', 'k': '2160p60'}
MEDIA = pathlib.Path('/content/media')


def render(level_dir, scene, q='h', show=True, save_to_drive=False):
    """level_dir: 'L01_is_it_honest'  ·  scene: 'L01Cinematic'  ·  q: l/m/h/k"""
    script = PROJECT / level_dir / 'scene.py'
    assert script.exists(), f'scene.py nahi mila: {script}'

    cmd = ['manim', f'-q{q}', '-v', 'WARNING', '--media_dir', str(MEDIA), str(script), scene]
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout[-4000:])
        print(proc.stderr[-4000:])
        raise RuntimeError('render fail — upar ka traceback dekhein')

    out = MEDIA / 'videos' / script.stem / QDIR[q] / f'{scene}.mp4'
    assert out.exists(), f'output nahi mila: {out}'
    print('OK  ', out, f'({out.stat().st_size / 1e6:.1f} MB)')

    if save_to_drive:
        dest = PROJECT / level_dir / 'renders'
        dest.mkdir(exist_ok=True)
        target = dest / f'{scene}_{QDIR[q]}.mp4'
        target.write_bytes(out.read_bytes())
        print('SAVED', target)

    if show:
        display(Video(str(out), embed=True, width=880))
    return out


print('render() ready.  Draft ke liye q="l", final ke liye q="h".')

---
# ACT I — FIRST CONTACT
## L01 · IS IT HONEST?

| Scene | Kya hai | Lambai |
|---|---|---|
| `L01Cinematic` | pura level — probe, glitch, definition, myths, repair, reward | ~3 min |
| `L01Myths` | sirf `many→1` vs `1→many` panel | ~25 sec |
| `L01Recap` | 5-point revision reel | ~40 sec |

Pehli baar `q='l'` (480p, tez) se dekh lein, sab theek lage to `q='h'`.

In [ ]:
render('L01_is_it_honest', 'L01Cinematic', q='l')

In [ ]:
render('L01_is_it_honest', 'L01Cinematic', q='h', save_to_drive=True)

In [ ]:
render('L01_is_it_honest', 'L01Myths', q='h')
render('L01_is_it_honest', 'L01Recap', q='h')

---
## L02 · NAME THE WIRES

| Scene | Kya hai | Lambai |
|---|---|---|
| `L02Cinematic` | pura level — machine diagram, f vs f(x), variables, arrow diagram, notation traps, f(a+h) | ~3.5 min |
| `L02Traps` | sirf `f(2a)` vs `2f(a)` aur `f(a^2)` vs `[f(a)]^2` | ~30 sec |
| `L02Recap` | 6-point revision reel | ~45 sec |

Khaas beat: **`f(a+h)`** — woh L10 (difference quotient) aur Chapter 2 ka darwaza hai.

In [ ]:
render('L02_name_the_wires', 'L02Cinematic', q='l')

In [ ]:
render('L02_name_the_wires', 'L02Cinematic', q='h', save_to_drive=True)
render('L02_name_the_wires', 'L02Traps', q='h')
render('L02_name_the_wires', 'L02Recap', q='h')

---
## Batch render — kisi bhi level ke saare scenes

`scene.py` me se scene names khud dhoond leta hai, phir sab render kar ke Drive me
`<level>/renders/` folder me save karta hai. Naye levels ke liye bas ek line add karein.

In [ ]:
import re


def scenes_in(level_dir):
    """scene.py me define ki gayi tamam Scene classes (file order me)."""
    src = (PROJECT / level_dir / 'scene.py').read_text(encoding='utf-8')
    return re.findall(r'^class\s+(\w+)\s*\(\s*(?:Moving)?Scene\s*\)', src, re.M)


def render_level(level_dir, q='h'):
    names = scenes_in(level_dir)
    print(level_dir, '->', names, '\n')
    for s in names:
        render(level_dir, s, q=q, show=False, save_to_drive=True)
    print(f'\nSab renders: {level_dir}/renders/\n')


render_level('L01_is_it_honest', q='h')
render_level('L02_name_the_wires', q='h')

---
## Plan B — Drive ke bagair (zip upload)

Cell 2 ki jagah yeh chalayein, phir `ch01-black-box.zip` upload karein.

In [ ]:
# from google.colab import files
# from pathlib import Path
# import zipfile, sys
#
# up = files.upload()                      # ch01-black-box.zip chunein
# name = list(up)[0]
# with zipfile.ZipFile(name) as z:
#     z.extractall('/content')
# PROJECT = Path('/content/ch01-black-box')
# sys.path.insert(0, str(PROJECT / 'shared'))
# print(PROJECT, sorted(p.name for p in PROJECT.glob('L*_*')))

---
## Troubleshooting

| Masla | Wajah / hal |
|---|---|
| `ModuleNotFoundError: theme` | Cell 2 nahi chala, ya `PROJECT` path galat hai. `shared/` folder saath upload hua? |
| `assert PROJECT.exists()` fail | Drive me folder ka naam exactly `ch01-black-box` rakhein, ya `PROJECT` line edit karein |
| Video blank / kaala | `q='l'` se try karein; agar phir bhi, `print(proc.stderr)` me error dekhein |
| Text ki jagah dabbe (□□□) | font missing — `!apt-get install -y fonts-dejavu` chalayein |
| Render bohat slow | `q='l'` use karein (480p). 1080p ka `L01Cinematic` ~3-5 min leta hai |
| `manim: command not found` | Cell 1 dobara chalayein (session reset ho gaya) |
| Emoji nazar nahi aata | scenes me emoji jaan-boojh kar nahi hai — Colab me emoji font nahi hota. Icons vector hain (`magnifier_icon()`) |

**Quality flags:** `-ql` 480p15 draft · `-qm` 720p30 · `-qh` 1080p60 · `-qk` 4K (Colab pe avoid karein)

**Docs:** https://docs.manim.community